# 02 · Pose extraction with MediaPipe Pose Landmarker

Extracts the 33 body landmarks for every frame of every clip and stores them as
Parquet, one file per video, with metadata recoverable from the path.

Decisions taken here (and why):

| Decision | Value | Reason |
|---|---|---|
| Model | `pose_landmarker_full.task` | same model the browser will load, so train/serve match |
| Running mode | `VIDEO` | measured 60% -> 100% detection rate vs `IMAGE` on hard side views |
| Confidence thresholds | 0.30 | higher recall; failed frames are kept and counted |
| Sampling | 10 fps (native 15 fps) | halves compute with no visible loss of angle resolution |
| Failed frames | kept, `detected=0` | so the failure rate is a measurable metric, not a silent gap |

## Shared setup (run first)

This notebook uses the modules in `ml/`. Upload them once to the Colab runtime:

1. Click the folder icon in the left sidebar (Files).
2. Click **Upload to session storage** and select `extract_pose.py`,
   `fitness_features.py`, `build_dataset.py`, `train_baseline.py`,
   `train_temporal.py`, `export_onnx.py` from `F:\Fitness Coach\ml`.
3. Run the cell below.

The dataset zip is expected at
`/content/drive/MyDrive/Dataset Exercise Quality-wise.zip` (Google Drive).

In [ ]:
import os, sys, subprocess, json, glob
from pathlib import Path

ROOT = Path('/content/fitness')
ROOT.mkdir(parents=True, exist_ok=True)

# The uploaded modules land in /content; mirror them into the work root.
for f in ['extract_pose.py', 'fitness_features.py', 'build_dataset.py', 'train_baseline.py',
          'experiment_generalization.py', 'experiment_clip.py', 'experiment_repmetry.py',
          'train_final.py', 'train_temporal.py', 'export_onnx.py']:
    src = Path('/content') / f
    if src.exists():
        dst = ROOT / f
        if not dst.exists() or src.stat().st_mtime > dst.stat().st_mtime:
            dst.write_bytes(src.read_bytes())

sys.path.insert(0, str(ROOT))
print('work root :', ROOT)
print('modules   :', sorted(p.name for p in ROOT.glob('*.py')))
print('cuda      :', subprocess.run('nvidia-smi -L', shell=True, capture_output=True, text=True).stdout.strip() or 'no GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/Dataset Exercise Quality-wise.zip'
import os
print('zip found:', os.path.exists(ZIP), round(os.path.getsize(ZIP)/1e9, 2), 'GB')

## 1. Extract only the three target exercises

In [ ]:
import zipfile, time, os
TARGET = {'squat', 'bicep_curl', 'push_up'}
RAW = ROOT / 'raw'
z = zipfile.ZipFile(ZIP)
entries = [n for n in z.namelist() if n.lower().endswith('.mp4') and n.split('/')[1] in TARGET]
print('videos to extract:', len(entries))
t0 = time.time(); done = 0
for n in entries:
    dest = RAW / '/'.join(n.split('/')[1:])
    if dest.exists() and dest.stat().st_size > 0:
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    with z.open(n) as src, open(dest, 'wb') as fh:
        while True:
            b = src.read(1 << 20)
            if not b:
                break
            fh.write(b)
    done += 1
print(f'extracted {done} files in {time.time()-t0:.0f}s -> {RAW}')

## 2. Download the pose model

In [ ]:
import urllib.request, os
MDL = ROOT / 'models'; MDL.mkdir(exist_ok=True)
url = 'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task'
dst = MDL / 'pose_landmarker_full.task'
if not dst.exists():
    urllib.request.urlretrieve(url, dst)
print('model:', dst, round(dst.stat().st_size/1e6, 2), 'MB')

import subprocess, sys
print(subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe'], capture_output=True, text=True).returncode)
import mediapipe as mp
print('mediapipe', mp.__version__)

## 3. Run the extraction

`extract_pose.py` uses 2 worker processes and one landmarker per video (the
VIDEO tracker is stateful and requires monotonically increasing timestamps).
Expect roughly 30-50 minutes for 444 clips on a free Colab CPU.

Output columns per row: `frame_idx, t_sec, sampled, detected, lm{k}_{x,y,z,v}`.

In [ ]:
!cd /content/fitness && WORKERS=2 python -u /content/fitness/extract_pose.py

## 4. Pose-detection availability report

In [ ]:
import json, pandas as pd
summary = json.load(open(ROOT / 'extract_summary.json'))
print('videos ok/failed:', summary['videos_ok'], '/', summary['videos_failed'])
print('frames sampled  :', summary['frames_sampled'])
print('overall detection rate:', round(summary['overall_detection_rate'], 4))
pv = pd.DataFrame(summary['per_video'])
print()
print(pv.detection_rate.describe().round(3))
print()
print('worst 10 clips:')
print(pv.nsmallest(10, 'detection_rate')[['path', 'detection_rate', 'frames_sampled']].to_string(index=False))

### What the failure rate means for the product

A clip where the pose is found in <60% of frames cannot be scored reliably, so
windows from those clips are dropped in notebook 03 (`min_detected_ratio`).
In the web app the same rule becomes: *"step back / improve lighting"* instead
of guessing.